# The Hybrid Model as a Diagnostic Instrument

So far we have used a learned closure to try to make a model *better*. This
notebook uses one to find out what is *wrong* with it.

The idea comes from Kalauni, Gupta & Bennett {cite}`kalauni2025hybrid`. Their
argument is that the interesting output of a hybrid model is not its score. It
is the answer to the question **"where did the network have to work hardest?"**
— because that is where your physics was failing. A hybrid model, set up
carefully, is an instrument for localizing structural error.

:::{admonition} Time check — 15 minutes
:class: tip
By the end you will be able to:
- run the same closure in several places and compare what each one buys
- read a learned function as a claim about a specific process
- explain why the *more constrained* hybrid often generalizes better
- say what this approach cannot yet do, which is the honest ending
:::

## Motivation

Calibration answers $\boldsymbol{\theta} = ?$. It cannot tell you that your ET
stress function has the wrong shape, because there is no parameter whose value
means "wrong shape". Shen et al. put the contrast crisply: traditional inversion
only ever asks $\boldsymbol{\theta} = ?$, while a differentiable model lets you
ask what the *function* should have been {cite}`shen2023differentiable`.

Kalauni et al. exploit this deliberately. They build three versions of a land
surface model that differ only in how much physics they keep:

| Configuration | What is learned | What is kept |
|---|---|---|
| Physics Only | nothing | everything, calibrated per site with CMA-ES |
| **Resistance NN** | aerodynamic $r_a$ and stomatal $r_s$ | the bulk-transfer equations for $H$ and $LE$ |
| **Full Flux NN** | $H$ and $LE$ directly | the ODE system advancing soil moisture and temperature |

That is a **spectrum of inductive bias** — designs C and D in the brief's
taxonomy — and the comparison between rungs is the measurement.

## What we'll cover

1. Three places to put the same closure
2. Running the experiment
3. Reading the scoreboard, including where it disagrees with itself
4. Reading the learned functions, which is more robust
5. What this cannot do yet

In [ ]:
# --- Colab bootstrap: installs the workshop package on first run -----------
try:
    import workshop_utils
except ImportError:
    %pip install -q "git+https://github.com/BennettHydroLab/differentiable_modeling_workshop.git"
    import workshop_utils

In [ ]:
import copy
import time

import numpy as np
import torch
import matplotlib.pyplot as plt

from workshop_utils import nse, summary, set_style, COLORS, ParamMap, MLP
from workshop_utils.data import (
    load_basin, split_by_water_year, BASIN_TEMPERATE, BASIN_LABELS,
)

set_style()
torch.manual_seed(0)

SPINUP = 365
S_REF = 10.0

## 1. Three question marks in one model

This is the HBV-lite from `03_hybrid_bucket_model`, with three closure slots
instead of one. Each slot replaces a different *constitutive relation* — a place
where the model says "the rate of this depends on the state of that", and where
somebody once chose a functional form.

| Slot | The physics it replaces | The hydrologic question it asks |
|---|---|---|
| `soil` | $R = P\,(S/FC)^{\beta}$ | how much rain becomes runoff, as a function of wetness? |
| `et` | $E = \mathrm{PET}\cdot\min(S/(LP\cdot FC),\,1)$ | how does evaporation shut down as the soil dries? |
| `route` | $q = K\,S$ | how fast does a store drain, as a function of how full it is? |

All three closures have the same form as in `03_hybrid_bucket_model`: the
network emits a **correction** to the physical quantity, inside a sigmoid, so
that a zero output reproduces the calibrated physics exactly and mass balance
is structural rather than learned. And all three take a single scalar input, so
all three can be plotted.

A fourth configuration turns on all three at once. That is our analogue of the
Full Flux NN — the least-constrained rung on the ladder.

In [ ]:
PARAM_NAMES = ["FC", "beta", "LP", "K_fast", "K_slow", "PERC"]
LO = torch.tensor([ 50., 1.0, 0.3, 0.05, 0.005, 0.1])
HI = torch.tensor([500., 5.0, 1.0, 0.60, 0.150, 3.0])
pmap = ParamMap(LO, HI)


def logit(p):
    return torch.log(p / (1 - p))


def simulate(P, E, theta, nets=None):
    """HBV-lite with up to three learned closures. nets: {'soil'|'et'|'route': MLP}."""
    nets = nets or {}
    FC, beta, LP, K_fast, K_slow, PERC = theta.T
    n = theta.shape[0]

    S_soil = torch.full((n,), 50.)
    S_fast = torch.full((n,), 5.)
    S_slow = torch.full((n,), 20.)

    out = []
    for t in range(P.shape[0]):
        wetness = torch.clamp(S_soil / FC, 0., 1.)

        # --- slot 1: runoff generation ---------------------------------------
        base = torch.clamp(wetness, 1e-4, 1 - 1e-4) ** beta
        if "soil" in nets:
            r_frac = torch.sigmoid(logit(base) + nets["soil"](wetness.unsqueeze(-1)).squeeze(-1))
        else:
            r_frac = base
        recharge = P[t] * r_frac

        # --- slot 2: evaporative stress --------------------------------------
        base = torch.clamp(S_soil / (LP * FC), 1e-4, 1 - 1e-4)
        if "et" in nets:
            e_frac = torch.sigmoid(logit(base) + nets["et"](wetness.unsqueeze(-1)).squeeze(-1))
        else:
            e_frac = base
        et = E[t] * e_frac

        S_soil = torch.clamp(S_soil + P[t] - recharge - et, min=0.)

        perc = torch.minimum(PERC, S_fast)
        S_fast = S_fast + recharge - perc

        # --- slot 3: storage-discharge ---------------------------------------
        if "route" in nets:
            k = torch.sigmoid(logit(K_fast)
                              + nets["route"]((S_fast / S_REF).unsqueeze(-1)).squeeze(-1))
        else:
            k = K_fast * torch.ones(n)
        q_fast = k * S_fast
        S_fast = S_fast - q_fast

        S_slow = S_slow + perc
        q_slow = K_slow * S_slow
        S_slow = S_slow - q_slow

        out.append(q_fast + q_slow)

    return torch.stack(out)

In [ ]:
ds = load_basin(BASIN_TEMPERATE)


def block(test_years):
    return split_by_water_year(ds, train=(1998, 1999), test=test_years, spinup_days=SPINUP)


def arrays(d):
    return tuple(torch.tensor(d[v].values, dtype=torch.float32)
                 for v in ["prcp", "pet", "qobs"])


P,  E,  Q  = arrays(block((2000, 2001))["train"])
Pv, Ev, Qv = arrays(block((2000, 2001))["test"])
Pt, Et, Qt = arrays(block((2002, 2004))["test"])

print(f"{BASIN_LABELS[BASIN_TEMPERATE]}: "
      f"{len(P)} train / {len(Pv)} validation / {len(Pt)} test days")

## 2. Running the experiment

Same protocol for every configuration, which is the entire point — if the
treatments differ in anything but the location of the closure, the comparison
means nothing.

- Start from the **same** calibrated physical parameters.
- Zero the output layer, so every hybrid begins numerically identical to the
  physics baseline.
- Same optimizer, same learning rates, same 15 gradient steps.
- Early-stop on validation; report on test.

In [ ]:
t0 = time.time()
raw_phys = torch.zeros(1, 6, requires_grad=True)
opt = torch.optim.Adam([raw_phys], lr=0.25)
for step in range(40):
    opt.zero_grad()
    loss = -nse(simulate(P, E, pmap(raw_phys))[SPINUP:, 0], Q[SPINUP:])
    loss.backward()
    opt.step()
raw_phys = raw_phys.detach()
theta_phys = pmap(raw_phys)
print(f"physics calibrated in {time.time()-t0:.1f} s, train NSE {-loss.item():.3f}")
print("  " + "  ".join(f"{n}={v:.3f}" for n, v in zip(PARAM_NAMES, theta_phys[0].tolist())))

In [ ]:
def evaluate(theta, nets=None, data=None):
    P_, E_, Q_ = data
    with torch.no_grad():
        q = simulate(P_, E_, theta, nets)[:, 0]
    return summary(q[SPINUP:], Q_[SPINUP:])


def train_hybrid(slots, steps=15, lr_phi=5e-3, seed=0):
    """Train one configuration. `slots` is a list like ['route'] or ['soil','et']."""
    torch.manual_seed(seed)
    nets = {s: MLP(n_in=1, n_out=1, hidden=16, depth=2) for s in slots}
    for net in nets.values():
        with torch.no_grad():          # start exactly at the calibrated physics
            net.net[-1].weight.zero_()
            net.net[-1].bias.zero_()

    raw = raw_phys.clone().requires_grad_(True)
    phi = [p for net in nets.values() for p in net.parameters()]
    opt = torch.optim.Adam([{"params": [raw], "lr": 0.05},
                            {"params": phi, "lr": lr_phi}])

    best, t0 = (-np.inf, None, 0), time.time()
    for step in range(steps):
        opt.zero_grad()
        loss = -nse(simulate(P, E, pmap(raw), nets)[SPINUP:, 0], Q[SPINUP:])
        loss.backward()
        torch.nn.utils.clip_grad_norm_([raw] + phi, 1.0)
        opt.step()
        v = evaluate(pmap(raw).detach(), nets, (Pv, Ev, Qv))["NSE"]
        if v > best[0]:
            best = (v, (raw.detach().clone(), copy.deepcopy(nets)), step + 1)

    raw_b, nets_b = best[1]
    return {
        "theta": pmap(raw_b), "nets": nets_b,
        "train": float(-loss.item()), "val": best[0], "step": best[2],
        "test": evaluate(pmap(raw_b), nets_b, (Pt, Et, Qt)),
        "seconds": time.time() - t0,
    }

In [ ]:
results = {"physics": {
    "theta": theta_phys, "nets": {},
    "train": float(-loss.item()),
    "val": evaluate(theta_phys, None, (Pv, Ev, Qv))["NSE"],
    "step": 40, "seconds": 0.0,
    "test": evaluate(theta_phys, None, (Pt, Et, Qt)),
}}

for slots, label in [(["soil"], "soil NN"),
                     (["et"], "ET NN"),
                     (["route"], "routing NN"),
                     (["soil", "et", "route"], "all three")]:
    results[label] = train_hybrid(slots)
    r = results[label]
    print(f"{label:12s}  {r['seconds']:5.1f} s   train {r['train']:.3f}   "
          f"val {r['val']:.3f}   test NSE {r['test']['NSE']:.3f}")

## 3. The scoreboard, and where it disagrees with itself

In [ ]:
cols = ["train NSE", "val NSE", "NSE", "KGE", "logNSE", "PBIAS", "RMSE"]
print(f"{'model':<13}" + "".join(f"{c:>11s}" for c in cols))
print("-" * (13 + 11 * len(cols)))
for name, r in results.items():
    row = [r["train"], r["val"], r["test"]["NSE"], r["test"]["KGE"],
           r["test"]["logNSE"], r["test"]["PBIAS"], r["test"]["RMSE"]]
    print(f"{name:<13}" + "".join(f"{v:11.3f}" for v in row))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), sharey=False)
names = list(results)
base_nse = results["physics"]["test"]["NSE"]
base_log = results["physics"]["test"]["logNSE"]

for ax, key, base, title in [
    (axes[0], "NSE", base_nse, "Held-out NSE — change from physics"),
    (axes[1], "logNSE", base_log, "Held-out logNSE — change from physics"),
]:
    vals = [results[n]["test"][key] - base for n in names]
    colors = [COLORS["physics"] if n == "physics" else
              (COLORS["hybrid"] if v > 0 else COLORS["ml"]) for n, v in zip(names, vals)]
    ax.bar(names, vals, color=colors)
    ax.axhline(0, color="k", lw=0.8)
    ax.set_ylabel(f"delta {key}")
    ax.set_title(title)
    ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()

Three things to take from that table, in order of how much you should trust them.

**First, and least comfortable: validation and test rank the configurations
differently.** Validation NSE — the only selection signal we are entitled to use
— likes the ET closure and the all-three model best. Test NSE says those two are
the *worst* of the four, and that only the routing closure improves on the
physics at all.

That is not a bug in the experiment, it is the experiment telling you something
important: **choosing between hybrid architectures needs more held-out data than
fitting one does.** Two water years of validation is enough to decide when to
stop training a given model; it is nowhere near enough to rank four models whose
scores differ by two hundredths. If you take one operational lesson from this
notebook, take that one.

**Second: log-NSE separates the configurations far more sharply than NSE does.**
The NSE column spans 0.03; the log-NSE column spans 0.20, and the ET closure
loses 0.17 of it. NSE is dominated by peaks and simply cannot see what these
models are doing differently. This is exactly the point both presenter papers
make independently — KGE cannot separate Lamichhane & Bennett's hybrid from a
pure LSTM while melt-out duration separates them by two weeks
{cite}`lamichhane2025dynamic`; NSE and KGEss rank Kalauni's two configurations in
opposite orders {cite}`kalauni2025hybrid`. **Always pair an aggregate score with
a process-specific diagnostic.**

**Third: turning on all three closures does not win.** The least-constrained
configuration gets the best training fit and lands in the middle on test. That
is the central finding of Kalauni et al., stated in their own words as the
thesis of the paper: physical inductive bias is *an asset for regional
generalization, not a handicap.* Their two hybrids are statistically
indistinguishable on every variance-based metric, and where they do differ — in
bias — the more constrained Resistance NN wins.

## 4. Reading the learned functions

Scores differing by hundredths are a weak instrument. The learned functions are
a much stronger one, and they are the reason we insisted on one-dimensional
closures. Each panel below shows what the network decided a constitutive
relation should look like, against the equation it replaced.

In [ ]:
w = torch.linspace(0.01, 0.99, 200)
s = torch.linspace(0.0, 16.0, 200)

fig, axes = plt.subplots(1, 3, figsize=(13.5, 4))

with torch.no_grad():
    # --- runoff generation
    th = results["soil NN"]["theta"]
    base = torch.clamp(w, 1e-4, 1 - 1e-4) ** th[0, 1]
    learned = torch.sigmoid(logit(base) + results["soil NN"]["nets"]["soil"](w.unsqueeze(-1)).squeeze(-1))
    axes[0].plot(w, base, color=COLORS["physics"], label=f"physics: (S/FC)^{float(th[0,1]):.2f}")
    axes[0].plot(w, learned, color=COLORS["hybrid"], label="learned")
    axes[0].set_xlabel("wetness  S_soil / FC"); axes[0].set_ylabel("fraction of P becoming recharge")
    axes[0].set_title("Runoff generation")

    # --- evaporative stress
    th = results["ET NN"]["theta"]
    base = torch.clamp(w * th[0, 0] / (th[0, 2] * th[0, 0]), 1e-4, 1 - 1e-4)
    learned = torch.sigmoid(logit(base) + results["ET NN"]["nets"]["et"](w.unsqueeze(-1)).squeeze(-1))
    axes[1].plot(w, base, color=COLORS["physics"], label=f"physics: min(S/(LP*FC), 1)")
    axes[1].plot(w, learned, color=COLORS["hybrid"], label="learned")
    axes[1].set_xlabel("wetness  S_soil / FC"); axes[1].set_ylabel("ET / PET")
    axes[1].set_title("Evaporative stress")

    # --- storage-discharge
    th = results["routing NN"]["theta"]
    learned = torch.sigmoid(logit(th[0, 3])
                            + results["routing NN"]["nets"]["route"]((s / S_REF).unsqueeze(-1)).squeeze(-1))
    axes[2].plot(s, np.full_like(s, float(th[0, 3])), color=COLORS["physics"],
                 label=f"physics: k = {float(th[0,3]):.3f}")
    axes[2].plot(s, learned, color=COLORS["hybrid"], label="learned k(S)")
    axes[2].set_xlabel("fast-store storage (mm)"); axes[2].set_ylabel("release fraction k (1/day)")
    axes[2].set_title("Storage-discharge")

for ax in axes:
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

Now we can say something specific about each, which is what a diagnostic is for.

**Runoff generation.** The learned curve sits *below* the calibrated beta-curve
through the middle of the wetness range and rejoins it near saturation. The
network is asking for a **more threshold-like** response than
$(S/FC)^{\beta}$ can produce even at $\beta \approx 4.6$: less runoff from
moderately wet soil, then a sharp turn-on. That is a recognizable claim — it is
what you would expect if runoff here is generated by saturation of a limited
contributing area rather than smoothly over the whole catchment. It also did not
transfer to the test years, so treat it as a hypothesis, not a finding.

**Evaporative stress.** The learned curve sits well *above* the linear ramp:
this catchment, the network says, keeps evaporating near potential far longer
into a drying cycle than $\min(S/(LP\cdot FC), 1)$ allows. Physically plausible —
real stress functions are usually concave, because deep roots keep transpiring
after the surface has dried. But this is also the configuration that cost the
most held-out skill, and it cost it specifically in log-NSE. Draw more ET in
summer and the soil store empties, so baseflow in the following months is too
low. **The learned function is defensible and the model it produces is worse.**
That combination is worth sitting with: a closure can be learning something real
about ET while being wrong about the water balance, because the only thing it
was scored on was discharge.

**Storage-discharge.** A gentle, monotone rise in release fraction with storage —
the convex storage-discharge relation from `03_hybrid_bucket_model`. It is the
smallest change of the three, it is the only one that improved held-out skill on
every variance-based metric, and its shape has a century of recession analysis
behind it.

**So where is this model's structural error?** The instrument's answer is: in
the routing, and that is where fixing it pays. The soil and ET relations both
have shapes the data prefer *in-sample*, and neither transfers. Compare with the
result Kalauni et al. get from the same logic: because their Resistance NN keeps
the bulk-transfer equations, the improvement is attributable purely to the
resistances, giving a specific actionable finding — stomatal resistance needs an
explicit vapour-pressure-deficit dependence, not just a soil-moisture one
{cite}`kalauni2025hybrid`. That is the shape of conclusion this method is for.

## 5. The honest limits

Four of them, and none is solved.

**The network may be fixing the solver, not the physics.** Our closure sits
inside a forward-Euler daily loop. Some of what it learned could be compensating
for the discretization rather than correcting the constitutive relation. Kalauni
et al. raise exactly this and do not resolve it. A cheap partial check: rerun at
a smaller timestep and see whether the learned function moves.

**Equifinality is undiminished.** Different random seeds land on different
learned functions with nearly identical scores. Beven's argument
{cite}`beven2006manifesto` applies to functions just as it does to parameters —
arguably more so, because a function has more ways to be wrong in compensating
directions.

**One basin is not evidence.** Everything here is Cowpasture River,
WY1998-2004. The diagnostic claim "the routing is where the structural error is"
is a claim about *this catchment*, and the way to test it is to run the same
four configurations across many basins at once — which, as
`00_setup_and_motivation` measured, costs essentially nothing extra because the
time loop is paid once regardless of batch width. Kalauni et al. run 135 sites;
Lamichhane & Bennett run 734.

**And the big one: we still have a function, not an equation.** We can plot
$k(S)$, we can see it is convex, we can fit a power law to it. What we cannot
yet do is *derive* the right equation and put it back into the model in a form
another modeler could adopt. Kalauni et al. name this precisely: "How to distill
them into interpretable and sensible equations, on the other hand, is an open
question that we take up in an upcoming study."

The available tooling — sparse symbolic regression on a UDE's learned term — is
real but not reliable. Rackauckas et al. recover Lotka-Volterra's missing
quadratic terms where SINDy on splined derivatives fails, and their own
robustness study puts the recovery rate at **(50.4 ± 25.7)%** across 498
error-free runs {cite}`rackauckas2021universal`. That is a coin flip, on a
two-species ODE with no observation noise.

`07_open_questions` takes it from here.

## Your turn

**Exercise 1 (3 minutes).** Rerun `train_hybrid(["route"])` with `seed=1`, `2`,
`3`. How much does the learned $k(S)$ curve move between seeds, and how much does
test NSE move? Which of the two is the more stable thing to report?

**Exercise 2 (5 minutes).** Change the training loss to
`0.5*(1 - nse(...)) + 0.5*(1 - log_nse(...))` and rerun all four configurations.
Does the ranking change? Does the learned ET stress function change?

:::{dropdown} Solution — Exercise 1
```python
for seed in [0, 1, 2, 3]:
    r = train_hybrid(["route"], seed=seed)
    print(seed, round(r["test"]["NSE"], 3))
```

Test NSE typically moves by 0.01-0.02 across seeds — the same order as the
difference between configurations in the table above. The *shape* of $k(S)$ —
monotone increasing, roughly a power law — is far more stable than its exact
height.

The lesson is what to report. A hybrid modeling result of the form "NSE improved
from 0.613 to 0.626" is barely distinguishable from seed noise. A result of the
form "the learned storage-discharge relation is consistently convex across
seeds" is a much stronger claim, and it is the kind this method is actually good
at producing.
:::

:::{dropdown} Solution — Exercise 2
```python
from workshop_utils import log_nse
# in train_hybrid, replace the loss line with:
q = simulate(P, E, pmap(raw), nets)[SPINUP:, 0]
loss = 0.5 * (1 - nse(q, Q[SPINUP:])) + 0.5 * (1 - log_nse(q, Q[SPINUP:]))
```

The ET closure stops being the worst, because the objective now penalizes the
low-flow damage it was doing for free. The routing closure's learned function
changes relatively little, because it was already being rewarded for recessions.

The general point is uncomfortable and worth stating plainly: **"where the
network helped" is a statement about your objective function as much as about
your model's physics.** A diagnostic built on a variance-based loss will find
structural errors that hurt peaks and will be blind to structural errors that
hurt water balance. Kalauni et al. hit the same wall from the other side — their
hybrids win the diurnal cycle and lose the long-term evaporative index, partly
because both were trained on variance-based losses that do not penalize bias.
:::

## Takeaways

- A hybrid model is an **instrument**, not just a better model. Put the same
  closure in different places, keep everything else identical, and the pattern
  of where learning helps localizes your structural error.
- **The more constrained hybrid generalized better here**, as in Kalauni et al.
  Turning on all three closures bought the best training fit and did not win on
  held-out data. Inductive bias is an asset.
- **Ranking architectures is much harder than fitting one.** Our validation block
  preferred two configurations that the test block ranked last. Differences of
  two hundredths of NSE across four models on two water years are not a result.
- **Read the learned function, not just the score.** It is more stable across
  seeds than the metrics are, and it is the only output of this whole exercise
  that another modeler can argue with.
- A learned function that is physically defensible can still make the model
  worse — our ET closure did. Being scored only on discharge is not the same as
  being right about evaporation.

## Where next

`06_practical_guide` collects the things that will bite you when you try this on
your own model. `07_open_questions` picks up the thread this notebook ends on:
how do you get from a plotted closure to a written equation?

## References

```{bibliography}
:filter: docname in docnames
```